# Eksperimen Machine Learning — Wine Quality Binary Classification

**Student:** M_Najwan_Naufal_A  
**Username Dicoding:** najwanopal  
**Course:** Membangun Sistem Machine Learning — Dicoding  
**Dataset:** Wine Quality Dataset (UCI Machine Learning Repository)  
**Task:** Binary Classification — Apakah wine berkualitas tinggi (quality ≥ 7) atau tidak

---

## 1. Import Libraries

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Section 1: Import Libraries
# ══════════════════════════════════════════════════════════════════════

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Serialization
import pickle
import os

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# ── Visualization settings ────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
%matplotlib inline

print('✅ Semua library berhasil di-import!')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')
print(f'   Sklearn : {__import__("sklearn").__version__}')

---
## 2. Data Loading

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Section 2: Data Loading
# ══════════════════════════════════════════════════════════════════════

# Path ke dataset combined (relatif dari folder preprocessing/)
DATA_PATH = os.path.join('..', 'winequality_raw', 'winequality_combined.csv')

# Load dataset
df = pd.read_csv(DATA_PATH)

print('=' * 60)
print('  WINE QUALITY DATASET — LOADED')
print('=' * 60)
print(f'\n📊 Shape: {df.shape[0]} baris × {df.shape[1]} kolom')
print(f'📦 Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Tampilkan data types setiap kolom
print('\n📋 Data Types:')
print('-' * 40)
for col in df.columns:
    print(f'  {col:<25s} {str(df[col].dtype):<10s}')
print(f'\n  Total kolom: {len(df.columns)}')

In [ ]:
# Tampilkan 10 baris pertama
print('\n🔍 Head (10 baris pertama):')
df.head(10)

In [ ]:
# Tampilkan 5 baris terakhir
print('\n🔍 Tail (5 baris terakhir):')
df.tail(5)

In [ ]:
# Info dataset lengkap
print('\n📝 Dataset Info:')
print('=' * 60)
df.info()

---
## 3. Exploratory Data Analysis (EDA)

Analisis eksplorasi data yang komprehensif untuk memahami karakteristik dataset sebelum preprocessing.

### 3a. Statistik Deskriptif

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3a. Statistik Deskriptif
# ══════════════════════════════════════════════════════════════════════

desc = df.describe().T
desc['range'] = desc['max'] - desc['min']
desc['iqr']   = desc['75%'] - desc['25%']
desc['cv']    = (desc['std'] / desc['mean'] * 100).round(2)  # Coefficient of Variation (%)

print('📊 Statistik Deskriptif (dengan Range, IQR, dan Coefficient of Variation):')
desc.round(4)

### 3b. Cek Missing Values

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3b. Cek Missing Values
# ══════════════════════════════════════════════════════════════════════

# Tabel missing values
missing = pd.DataFrame({
    'Kolom': df.columns,
    'Missing Count': df.isnull().sum().values,
    'Missing %': (df.isnull().sum().values / len(df) * 100).round(2)
})
print('🔍 Tabel Missing Values:')
print(missing.to_string(index=False))
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

# Heatmap missing values
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=ax)
ax.set_title('Heatmap Missing Values', fontsize=14, fontweight='bold')
ax.set_xlabel('Kolom')
ax.set_ylabel('Baris')
plt.tight_layout()
plt.show()

print('\n✅ Tidak ada missing values dalam dataset!')

### 3c. Cek Duplikasi Data

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3c. Cek Duplikasi Data
# ══════════════════════════════════════════════════════════════════════

n_duplicates = df.duplicated().sum()
pct_duplicates = n_duplicates / len(df) * 100

print(f'🔁 Jumlah baris duplikat  : {n_duplicates}')
print(f'   Persentase duplikat    : {pct_duplicates:.2f}%')
print(f'   Baris unik             : {len(df) - n_duplicates}')

if n_duplicates > 0:
    print(f'\n⚠️  Ditemukan {n_duplicates} baris duplikat. Akan di-handle pada tahap preprocessing.')
    print('\nContoh baris duplikat:')
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))

### 3d. Distribusi Target Variable (Class Imbalance Check)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3d. Distribusi Target Variable
# ══════════════════════════════════════════════════════════════════════

# Buat kolom binary target sementara untuk analisis
df['_quality_label'] = (df['quality'] >= 7).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Distribusi quality score asli ---
quality_counts = df['quality'].value_counts().sort_index()
colors_q = plt.cm.viridis(np.linspace(0.2, 0.9, len(quality_counts)))
axes[0].bar(quality_counts.index, quality_counts.values, color=colors_q, edgecolor='black', linewidth=0.5)
for i, (q, c) in enumerate(zip(quality_counts.index, quality_counts.values)):
    axes[0].text(q, c + 30, str(c), ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Distribusi Quality Score (Asli)', fontweight='bold')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Jumlah')

# --- Plot 2: Distribusi binary target ---
binary_counts = df['_quality_label'].value_counts().sort_index()
labels_bin = ['Low Quality\n(< 7)', 'High Quality\n(≥ 7)']
colors_bin = ['#e74c3c', '#2ecc71']
bars = axes[1].bar(labels_bin, binary_counts.values, color=colors_bin, edgecolor='black', linewidth=0.5)
for bar, count in zip(bars, binary_counts.values):
    pct = count / len(df) * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Distribusi Binary Target', fontweight='bold')
axes[1].set_ylabel('Jumlah')

# --- Plot 3: Pie chart ---
axes[2].pie(binary_counts.values, labels=['Low Quality (< 7)', 'High Quality (≥ 7)'],
            colors=colors_bin, autopct='%1.1f%%', startangle=90,
            explode=(0, 0.05), shadow=True, textprops={'fontsize': 11})
axes[2].set_title('Proporsi Kelas Target', fontweight='bold')

plt.suptitle('Analisis Distribusi Target Variable', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Imbalance ratio
ratio = binary_counts[0] / binary_counts[1]
print(f'\n⚖️  Imbalance Ratio: {ratio:.2f} : 1')
print(f'     Low Quality  (0): {binary_counts[0]}  ({binary_counts[0]/len(df)*100:.1f}%)')
print(f'     High Quality (1): {binary_counts[1]}  ({binary_counts[1]/len(df)*100:.1f}%)')
print(f'\n⚠️  Dataset IMBALANCED — kelas minoritas (High Quality) hanya ~{binary_counts[1]/len(df)*100:.0f}%')

# Hapus kolom sementara
df.drop(columns=['_quality_label'], inplace=True)

### 3e. Distribusi Setiap Fitur Numerik (Histogram + KDE)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3e. Distribusi Setiap Fitur Numerik (Histogram + KDE)
# ══════════════════════════════════════════════════════════════════════

# Semua kolom numerik kecuali quality (target)
numeric_features = [col for col in df.select_dtypes(include=[np.number]).columns
                    if col != 'quality']

n_features = len(numeric_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(numeric_features):
    ax = axes[i]
    # Histogram dengan KDE overlay
    sns.histplot(data=df, x=feature, kde=True, ax=ax, color='steelblue',
                 edgecolor='black', linewidth=0.5, alpha=0.7, bins=40)
    
    # Tambahkan garis mean dan median
    mean_val = df[feature].mean()
    median_val = df[feature].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_val:.2f}')
    ax.axvline(median_val, color='green', linestyle='-', linewidth=1.5, label=f'Median: {median_val:.2f}')
    
    ax.set_title(feature, fontweight='bold', fontsize=12)
    ax.legend(fontsize=8)

# Sembunyikan axes kosong
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribusi Fitur Numerik (Histogram + KDE)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3f. Boxplot per Fitur untuk Deteksi Outlier

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3f. Boxplot per Fitur untuk Deteksi Outlier
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(numeric_features):
    ax = axes[i]
    sns.boxplot(data=df, y=feature, ax=ax, color='lightblue',
                flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
    
    # Hitung jumlah outlier menggunakan IQR
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[feature] < lower) | (df[feature] > upper)).sum()
    
    ax.set_title(f'{feature}\n(outliers: {n_outliers})', fontweight='bold', fontsize=11)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplot per Fitur — Deteksi Outlier',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Tampilkan tabel ringkasan outlier
print('\n📋 Ringkasan Outlier per Fitur (IQR Method):')
print('-' * 55)
print(f'{"Fitur":<28s} {"Outliers":>8s} {"% Data":>8s}')
print('-' * 55)
for feature in numeric_features:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[feature] < lower) | (df[feature] > upper)).sum()
    pct = n_outliers / len(df) * 100
    print(f'  {feature:<26s} {n_outliers:>6d}   {pct:>6.2f}%')
print('-' * 55)

### 3g. Correlation Matrix Heatmap

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3g. Correlation Matrix Heatmap
# ══════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(14, 11))

# Korelasi semua kolom numerik
corr_matrix = df.select_dtypes(include=[np.number]).corr()

# Mask segitiga atas agar tidak redundan
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            vmin=-1, vmax=1, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Korelasi Pearson'})

ax.set_title('Correlation Matrix Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Korelasi terhadap quality (target)
print('\n📊 Korelasi fitur terhadap quality (diurutkan):')
print('-' * 45)
corr_target = corr_matrix['quality'].drop('quality').sort_values(ascending=False)
for feat, corr_val in corr_target.items():
    bar = '█' * int(abs(corr_val) * 30)
    sign = '+' if corr_val > 0 else '-'
    print(f'  {feat:<25s} {sign}{abs(corr_val):.4f}  {bar}')

### 3h. Pairplot Fitur-Fitur Penting

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3h. Pairplot Fitur-Fitur Penting
# ══════════════════════════════════════════════════════════════════════

# Pilih top 4 fitur berdasarkan korelasi terhadap quality + target
top_features = corr_matrix['quality'].drop('quality').abs().sort_values(ascending=False).head(4).index.tolist()
print(f'Top 4 fitur berkorelasi tinggi dengan quality: {top_features}')

# Buat kolom binary sementara untuk pewarnaan
df_plot = df.copy()
df_plot['Quality Label'] = df_plot['quality'].apply(lambda x: 'High (≥7)' if x >= 7 else 'Low (<7)')

g = sns.pairplot(df_plot[top_features + ['Quality Label']],
                 hue='Quality Label',
                 palette={'Low (<7)': '#e74c3c', 'High (≥7)': '#2ecc71'},
                 diag_kind='kde',
                 plot_kws={'alpha': 0.4, 's': 15},
                 height=2.5)
g.figure.suptitle('Pairplot Fitur Penting vs Quality Label',
                  y=1.02, fontsize=16, fontweight='bold')
plt.show()

df_plot.drop(columns=['Quality Label'], inplace=True)

### 3i. Analisis Wine Type vs Quality

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3i. Analisis Wine Type vs Quality
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

type_labels = {0: 'Red Wine', 1: 'White Wine'}
type_colors = ['#c0392b', '#f1c40f']

# --- Plot 1: Quality score distribution by wine type ---
for wt, label in type_labels.items():
    subset = df[df['wine_type'] == wt]['quality']
    axes[0].hist(subset, bins=range(3, 10), alpha=0.6, label=label,
                 color=type_colors[wt], edgecolor='black', linewidth=0.5)
axes[0].set_title('Quality Score by Wine Type', fontweight='bold')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# --- Plot 2: Stacked bar chart ---
ct = pd.crosstab(df['quality'], df['wine_type'].map(type_labels))
ct.plot(kind='bar', stacked=True, color=type_colors, ax=axes[1],
        edgecolor='black', linewidth=0.5)
axes[1].set_title('Quality Distribution (Stacked)', fontweight='bold')
axes[1].set_xlabel('Quality Score')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

# --- Plot 3: Persentase High Quality per Wine Type ---
pct_data = []
for wt, label in type_labels.items():
    subset = df[df['wine_type'] == wt]
    high_pct = (subset['quality'] >= 7).mean() * 100
    pct_data.append({'Wine Type': label, 'High Quality %': high_pct})

pct_df = pd.DataFrame(pct_data)
bars = axes[2].bar(pct_df['Wine Type'], pct_df['High Quality %'],
                   color=type_colors, edgecolor='black', linewidth=0.5)
for bar, pct in zip(bars, pct_df['High Quality %']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{pct:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[2].set_title('% High Quality (≥7) per Wine Type', fontweight='bold')
axes[2].set_ylabel('Percentage (%)')

plt.suptitle('Analisis Wine Type vs Quality', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Statistik per wine type
print('\n📊 Statistik per Wine Type:')
for wt, label in type_labels.items():
    subset = df[df['wine_type'] == wt]
    high = (subset['quality'] >= 7).sum()
    print(f'  {label:<12s}: {len(subset):>5d} samples, '
          f'High Quality: {high:>4d} ({high/len(subset)*100:.1f}%), '
          f'Avg Quality: {subset["quality"].mean():.2f}')

### 3j. Insight dan Kesimpulan EDA

**Temuan utama dari Exploratory Data Analysis:**

1. **Dataset Shape:** 6497 sampel dengan 13 kolom (11 fitur fisikokimia + wine_type + quality target)

2. **Missing Values:** Tidak ada missing values — dataset sudah bersih

3. **Duplikasi:** Terdapat beberapa baris duplikat yang perlu di-handle

4. **Class Imbalance:** Dataset sangat imbalanced!
   - Low Quality (< 7): ~80.3% (5220 sampel)
   - High Quality (≥ 7): ~19.7% (1277 sampel)
   - Imbalance ratio ~4:1 → perlu teknik oversampling (SMOTE) saat training

5. **Fitur Berkorelasi Tinggi dengan Quality:**
   - `alcohol` (korelasi positif tertinggi)
   - `volatile acidity` (korelasi negatif tertinggi)
   - `density` (korelasi negatif)
   - `sulphates` dan `citric acid` (korelasi positif sedang)

6. **Outlier:** Beberapa fitur memiliki outlier signifikan, terutama:
   - `residual sugar`, `chlorides`, `free sulfur dioxide`, `total sulfur dioxide`
   - Akan di-handle dengan IQR capping pada preprocessing

7. **Wine Type:** White wine memiliki proporsi high quality yang sedikit berbeda dari red wine

8. **Distribusi Fitur:** Beberapa fitur memiliki distribusi skewed (right-skewed), yang wajar untuk data kimia

**Keputusan Preprocessing:**
- Drop duplikat
- Handle outlier dengan IQR capping
- Encode wine_type dengan LabelEncoder
- StandardScaler untuk normalisasi fitur
- Stratified train-test split (80:20) untuk menjaga proporsi kelas

---
## 4. Data Preprocessing

Langkah-langkah preprocessing untuk menyiapkan data sebelum modeling.

In [ ]:
# Simpan shape awal sebelum preprocessing
shape_before = df.shape
print(f'📊 Shape SEBELUM preprocessing: {shape_before}')
print(f'   Jumlah sampel: {shape_before[0]}')
print(f'   Jumlah kolom : {shape_before[1]}')

### 4a. Handling Duplicates

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4a. Handling Duplicates
# ══════════════════════════════════════════════════════════════════════

n_before = len(df)
n_dups = df.duplicated().sum()

print(f'Baris sebelum drop: {n_before}')
print(f'Baris duplikat    : {n_dups}')

# Drop duplikat
df = df.drop_duplicates().reset_index(drop=True)

n_after = len(df)
print(f'Baris setelah drop: {n_after}')
print(f'\n✅ Berhasil menghapus {n_before - n_after} baris duplikat')

### 4b. Handling Missing Values

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4b. Handling Missing Values
# ══════════════════════════════════════════════════════════════════════

missing_total = df.isnull().sum().sum()
print(f'Total missing values: {missing_total}')

if missing_total == 0:
    print('\n✅ Tidak ada missing values — tidak perlu handling tambahan')
else:
    # Jika ada missing values, isi dengan median
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f'  {col}: filled {df[col].isnull().sum()} missing with median={median_val:.4f}')
    print(f'\n✅ Missing values di-handle dengan median imputation')

### 4c. Feature Engineering

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4c. Feature Engineering
# ══════════════════════════════════════════════════════════════════════

# --- 4c-1: Buat kolom target binary ---
# quality >= 7 → 1 (High Quality), else → 0 (Low Quality)
df['quality_label'] = (df['quality'] >= 7).astype(int)

print('📋 Kolom target binary "quality_label" dibuat:')
print(f'   High Quality (1): {(df["quality_label"] == 1).sum()}')
print(f'   Low Quality  (0): {(df["quality_label"] == 0).sum()}')

# --- 4c-2: Encode wine_type menggunakan LabelEncoder ---
le = LabelEncoder()
df['wine_type'] = le.fit_transform(df['wine_type'])

print(f'\n📋 wine_type di-encode dengan LabelEncoder:')
print(f'   Classes: {le.classes_}')
print(f'   Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Tampilkan dataframe setelah feature engineering
print(f'\n📊 Shape setelah feature engineering: {df.shape}')
print(f'\nKolom sekarang: {list(df.columns)}')
df.head()

### 4d. Handling Outliers (IQR Method)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4d. Handling Outliers menggunakan IQR Method (Capping/Winsorizing)
# ══════════════════════════════════════════════════════════════════════

# Fitur numerik yang perlu di-handle outliernya
# (tidak termasuk quality, quality_label, wine_type)
outlier_features = [col for col in df.select_dtypes(include=[np.number]).columns
                    if col not in ('quality', 'quality_label', 'wine_type')]

print('📋 Handling Outliers (IQR Capping Method):')
print('=' * 65)
print(f'{"Fitur":<28s} {"Lower":>8s} {"Upper":>8s} {"Capped":>8s}')
print('-' * 65)

total_capped = 0
for feature in outlier_features:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Hitung jumlah outlier sebelum capping
    n_outliers = ((df[feature] < lower_bound) | (df[feature] > upper_bound)).sum()
    
    # Capping: clip values ke batas IQR
    df[feature] = df[feature].clip(lower=lower_bound, upper=upper_bound)
    
    total_capped += n_outliers
    if n_outliers > 0:
        print(f'  {feature:<26s} {lower_bound:>8.3f} {upper_bound:>8.3f} {n_outliers:>6d}')

print('-' * 65)
print(f'  {"TOTAL CAPPED":<26s} {"": >8s} {"": >8s} {total_capped:>6d}')
print(f'\n✅ Outlier handling selesai — {total_capped} nilai di-cap ke batas IQR')

In [ ]:
# Visualisasi boxplot SETELAH handling outlier
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(numeric_features):
    if feature in df.columns:
        ax = axes[i]
        sns.boxplot(data=df, y=feature, ax=ax, color='lightgreen',
                    flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
        ax.set_title(feature, fontweight='bold', fontsize=11)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplot SETELAH Outlier Handling (IQR Capping)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4e. Feature Scaling (StandardScaler)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4e. Feature Scaling menggunakan StandardScaler
# ══════════════════════════════════════════════════════════════════════

# Pisahkan fitur (X) dan target (y)
# Drop kolom 'quality' (asli) karena kita pakai 'quality_label' sebagai target
feature_columns = [col for col in df.columns if col not in ('quality', 'quality_label')]

X = df[feature_columns].copy()
y = df['quality_label'].copy()

print(f'📊 Fitur (X): {X.shape}')
print(f'   Kolom: {list(X.columns)}')
print(f'\n📊 Target (y): {y.shape}')
print(f'   Distribusi: 0={( y==0).sum()}, 1={(y==1).sum()}')

### 4f. Train-Test Split (80:20, Stratified)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4f. Train-Test Split (80:20, Stratified, random_state=42)
# ══════════════════════════════════════════════════════════════════════

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print('📊 Hasil Train-Test Split (80:20, Stratified):')
print('=' * 50)
print(f'  X_train shape: {X_train.shape}')
print(f'  X_test  shape: {X_test.shape}')
print(f'  y_train shape: {y_train.shape}')
print(f'  y_test  shape: {y_test.shape}')

print(f'\n  Train set — Low: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%), '
      f'High: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)')
print(f'  Test  set — Low: {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%), '
      f'High: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)')

In [ ]:
# --- Apply StandardScaler ---
# PENTING: fit pada train, transform pada train dan test
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_columns,
    index=X_test.index
)

print('📊 Statistik X_train setelah scaling:')
print(f'   Mean  ≈ {X_train_scaled.mean().mean():.6f}  (mendekati 0)')
print(f'   Std   ≈ {X_train_scaled.std().mean():.6f}  (mendekati 1)')
print(f'\n✅ StandardScaler berhasil di-apply!')

# Preview data setelah scaling
print('\n🔍 Preview X_train setelah scaling:')
X_train_scaled.head()

### 4g. Simpan Hasil Preprocessing

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4g. Simpan Hasil Preprocessing
#  Output: preprocessing/winequality_preprocessing/
# ══════════════════════════════════════════════════════════════════════

# Buat folder output
OUTPUT_DIR = os.path.join('.', 'winequality_preprocessing')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Simpan dataset ---
X_train_scaled.to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
X_test_scaled.to_csv(os.path.join(OUTPUT_DIR, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(OUTPUT_DIR, 'y_test.csv'), index=False)

# --- Simpan scaler ---
with open(os.path.join(OUTPUT_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# --- Simpan label encoder ---
with open(os.path.join(OUTPUT_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

# Verifikasi
print('✅ Semua file preprocessing berhasil disimpan!')
print(f'\n📁 Output directory: {os.path.abspath(OUTPUT_DIR)}')
print('-' * 50)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  📄 {fname:<25s} ({size_kb:.1f} KB)')
print('-' * 50)

---
## 5. Kesimpulan

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Section 5: Kesimpulan — Ringkasan Preprocessing
# ══════════════════════════════════════════════════════════════════════

print('=' * 65)
print('  RINGKASAN PREPROCESSING')
print('=' * 65)

print(f'\n📊 Shape Data:')
print(f'  Sebelum preprocessing : {shape_before}')
print(f'  Setelah preprocessing : ({len(X_train_scaled) + len(X_test_scaled)}, {X_train_scaled.shape[1]})')
print(f'  Baris dihapus (dupl.) : {shape_before[0] - (len(X_train_scaled) + len(X_test_scaled))}')

print(f'\n📐 Split Dataset:')
print(f'  Train set: {X_train_scaled.shape[0]} sampel ({X_train_scaled.shape[0]/(len(X_train_scaled)+len(X_test_scaled))*100:.0f}%)')
print(f'  Test  set: {X_test_scaled.shape[0]} sampel ({X_test_scaled.shape[0]/(len(X_train_scaled)+len(X_test_scaled))*100:.0f}%)')

print(f'\n⚖️  Distribusi Kelas pada TRAIN SET:')
print(f'  Low Quality  (0): {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)')
print(f'  High Quality (1): {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)')

print(f'\n⚖️  Distribusi Kelas pada TEST SET:')
print(f'  Low Quality  (0): {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)')
print(f'  High Quality (1): {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)')

print(f'\n📋 Langkah Preprocessing yang Dilakukan:')
steps = [
    '1. Drop baris duplikat',
    '2. Cek & handle missing values (tidak ada)',
    '3. Feature engineering: kolom target binary (quality_label)',
    '4. Encode wine_type dengan LabelEncoder',
    '5. Handle outlier dengan IQR capping',
    '6. StandardScaler (fit pada train, transform pada train & test)',
    '7. Stratified train-test split (80:20, random_state=42)',
    '8. Simpan hasil ke preprocessing/winequality_preprocessing/',
]
for step in steps:
    print(f'  ✅ {step}')

print(f'\n📁 File yang Disimpan:')
saved_files = [
    ('X_train.csv', 'Fitur training (scaled)'),
    ('X_test.csv', 'Fitur testing (scaled)'),
    ('y_train.csv', 'Label training'),
    ('y_test.csv', 'Label testing'),
    ('scaler.pkl', 'StandardScaler (fitted)'),
    ('label_encoder.pkl', 'LabelEncoder (fitted)'),
]
for fname, desc in saved_files:
    print(f'  📄 {fname:<25s} — {desc}')

print(f'\n{"=" * 65}')
print('  ✅ PREPROCESSING SELESAI — Data siap untuk modeling!')
print(f'{"=" * 65}')

---

### Catatan:
- Data telah diproses dan disimpan di `preprocessing/winequality_preprocessing/`
- Scaler dan encoder disimpan untuk digunakan kembali saat inference
- Stratified split menjaga proporsi kelas yang sama di train dan test
- Outlier di-cap (bukan dihapus) untuk mempertahankan jumlah sampel
- Untuk menangani class imbalance, gunakan SMOTE saat training (bukan di preprocessing)

**Next Step:** Gunakan data ini untuk melatih model di fase berikutnya (Modeling & Eksperimen MLflow)